# DSA RAG Evaluation

Evaluate retrieval and QA on DSA documentation.

Steps:
- Inventory DSA document sources.
- Run retrieval evaluation.
- Issue a sample DSA query.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'dsa_docs': {},
    'eval_exit': None,
}


def run_optional(cmd: list[str]) -> int:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)
    print('Return code:', result.returncode)
    return result.returncode


In [ ]:
# Inventory DSA docs.
dsa_dir = REPO_ROOT / 'data' / 'dsa_docs'
if dsa_dir.exists():
    files = [p for p in dsa_dir.rglob('*') if p.is_file()]
    summary['dsa_docs']['file_count'] = len(files)
    print('DSA docs files:', len(files))
else:
    print('Missing:', dsa_dir)


In [ ]:
# Run retrieval evaluation.
script_path = REPO_ROOT / 'scripts' / 'rag' / 'evaluate_dsa.py'
if script_path.exists():
    summary['eval_exit'] = run_optional([PY, 'scripts/rag/evaluate_dsa.py'])
else:
    print('Missing:', script_path)


In [ ]:
# Run a quick DSA query.
try:
    from app.rag.dsa_pipeline import answer_query
except Exception as exc:
    answer_query = None
    print('Could not import DSA RAG pipeline:', exc)

if answer_query:
    try:
        result = answer_query('Explain binary search time complexity.')
        print(result.get('answer'))
        print('Sources:', len(result.get('sources', [])))
    except Exception as exc:
        print('Query failed:', exc)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_dsa_rag_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
